# Proposed system: fine-tuned usage encoder + sense clustering

Runs the proposed system on the shared split and scores it with `src/evaluate.py`.

| Step | What | Time on a free T4 |
|---|---|---|
| 1-4 | Set up: Drive, code, synthetic data, tests | ~3 min |
| 5 | Quick check on one fold with small settings (catches errors) | ~5 min |
| 6 | **Full run, seed 42** (Stage A synthetic + Stage B for 10 folds) | ~45-75 min |
| 7 | Frozen XLM-R through the same pipeline (ablation) | ~3 min |
| 8 | Clustering / Subtask 2 ablations on the saved vectors (no training) | ~3 min |
| 9-10 | Results table, download | ~1 min |

**Before running:** Runtime > Change runtime type > **T4 GPU**. Keep this browser tab open while it runs.

Your Google Drive must contain:
```
MyDrive/SenseShift/
    SenseShift-code.zip     (the repo code)
    sentences.zip           (synthetic English set from Codabench)
```
Checkpoints and results are written to `MyDrive/SenseShift/`, so nothing is lost if Colab disconnects.
Re-running step 6 after a disconnect re-uses the saved Stage A checkpoint.

In [ ]:
# Cell 1: connect Google Drive and check the input files
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/SenseShift'
for f in ['SenseShift-code.zip', 'sentences.zip']:
    path = f'{DRIVE}/{f}'
    print(('OK      ' if os.path.exists(path) else 'MISSING ') + path)
CKPT = f'{DRIVE}/checkpoints'     # Stage A checkpoint (about 1.1 GB)
RUNS = f'{DRIVE}/runs'            # one folder per run
os.makedirs(CKPT, exist_ok=True); os.makedirs(RUNS, exist_ok=True)

In [ ]:
# Cell 2: unpack the code, install packages, check the GPU
%cd /content
!rm -rf /content/SenseShift && mkdir -p /content/SenseShift
!unzip -q -o "{DRIVE}/SenseShift-code.zip" -d /content/SenseShift
%cd /content/SenseShift
!ls
!pip install -q bcubed nltk pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime > Change runtime type > T4 GPU')

The synthetic English set goes to `data/external/synthetic_en/generated_sentences/` (training only, never evaluated).

In [ ]:
# Cell 3: unpack the synthetic English training set
!mkdir -p data/external/synthetic_en
!unzip -q -o "{DRIVE}/sentences.zip" -d data/external/synthetic_en
!echo "word files:" $(find data/external/synthetic_en/generated_sentences -name '*.jsonl' | wc -l) "(expected 2347)"

Tests of the scorer, the data pipeline and the proposed system. The last line should say `37 passed`.

In [ ]:
# Cell 4: tests
!python -m pytest tests -q 2>&1 | tail -1

## Quick check (one fold, small settings)
Fold 0 (test word *dumpa*), 100 synthetic words, a few hundred pairs. The scores mean nothing; this only
checks that everything runs on the GPU before the long run. The first run downloads XLM-R (~1 GB).

In [ ]:
# Cell 5: quick check
!python -m src.proposed run --name quickcheck \
    --set "split.folds=[0]" --set training.synthetic.max_words=100 --set training.synthetic.n_pairs=2000 \
    --set training.epochs=1 --set training.pairs_per_epoch=500 \
    --set output.checkpoints_dir=/content/tmp_ckpt --set output.results_dir=/content/tmp_runs

## Full run: proposed system, seed 42
Stage A: contrastive training on ~50,000 synthetic English pairs (once, saved to Drive).
Stage B: for each of the 10 folds, fine-tune on the 8 training words, early-stop on the validation word,
encode all usages, cluster the test word. All settings come from `configs/default.yaml`.

In [ ]:
# Cell 6: full run (about 45-75 min)
!python -m src.proposed run --set seed=42 \
    --set output.checkpoints_dir="{CKPT}" --set output.results_dir="{RUNS}"

## Ablation: frozen encoder, same pipeline
Frozen XLM-R (layer 8, no markers, no training) with exactly the same clustering selection and Subtask 2
rule. Comparing this with Cell 6 isolates the effect of fine-tuning.

In [ ]:
# Cell 7: frozen encoder
!python -m src.proposed run --set training.enabled=false --set output.results_dir="{RUNS}"

## Ablations on the saved vectors (no retraining)
Re-use the vectors of the full run with other clustering selections and Subtask 2 rules.

In [ ]:
# Cell 8: clustering and Subtask 2 ablations
import glob
FULL = sorted(glob.glob(f'{RUNS}/*_proposed_ft_syn_sv_seed42'))[-1]
print('vectors from', FULL)
for sel in ['nine_words', 'fixed', 'validation_bcubed']:
    !python -m src.proposed recluster --run "{FULL}" --set clustering.threshold_selection={sel} --set output.results_dir="{RUNS}" 2>&1 | grep -E "Evaluation:|1a  |1b  |2   macro"
for mode in ['rank', 'cluster']:
    !python -m src.proposed recluster --run "{FULL}" --set subtask2.mode={mode} --set output.results_dir="{RUNS}" 2>&1 | grep -E "Evaluation:|2   macro"

## Results next to the baselines

In [ ]:
# Cell 9: summary table
import json, glob, os
import pandas as pd
rows = []
for f in sorted(glob.glob(f'{RUNS}/*/scores.json')):
    d = json.load(open(f)); s = d['scores']
    sh = d.get('subtask1', {}).get('subtask1b', {}).get('shuffle_control') or {}
    rows.append({'run': os.path.basename(os.path.dirname(f)), '1a B3 F1': s.get('1a_bcubed_f1'),
                 '1b rho': s.get('1b_spearman'), '1b shuffle': sh.get('shuffled_mean'),
                 '2 macro F1': s.get('2_macro_f1'), 'APD rho (local)': s.get('graded_spearman')})
ours = pd.DataFrame(rows).round(3)
base = pd.read_csv('results/baselines/summary.csv')[['system', '1a_bcubed_f1', '1b_spearman', '2_macro_f1', '1b_shuffle_control']]
print('Baselines (official scorer, same split):'); display(base)
print('Proposed system runs:'); display(ours)
ours.to_csv(f'{RUNS}/proposed_summary.csv', index=False)

## Download
`proposed_results.zip` holds every run folder (predictions, scores, config, selection, training history)
**without** the `vectors/` folders, which stay on Drive. Put the run folders into `results/runs/` of the repo.

In [ ]:
# Cell 10: zip and download
!cd "{RUNS}" && rm -f /content/proposed_results.zip && zip -qr /content/proposed_results.zip . -x "*/vectors/*"
from google.colab import files
files.download('/content/proposed_results.zip')

## More seeds (Section 10: mean ± sd over 5 seeds)
Each seed retrains Stage A (~10 min) and the 10 folds. Run one or two seeds per Colab session; the results
land in the same Drive folder. Then `python -m src.evaluate summarise` gives mean ± sd.

In [ ]:
# Cell 11: one more seed (change the number: 13, 87, 1234, 2027)
SEED = 13
!python -m src.proposed run --set seed={SEED} \
    --set output.checkpoints_dir="{CKPT}" --set output.results_dir="{RUNS}"